# Tactical GNN — 실제 데이터 v2

**v1 문제점 → v2 수정**
1. 라벨 4.8% → **전술 구간 내 프레임만 샘플링** (라벨률 ~100%)
2. UW 발산 → **log_vars 클램핑** (-2 ~ 2)
3. 클래스 편향 → **클래스 가중치** CrossEntropyLoss
4. 메모리 → **경기별 lazy load + 구간 내만 추출**

In [ ]:
!nvidia-smi
import torch
TORCH = torch.__version__.split('+')[0]
CUDA = torch.version.cuda.replace('.', '')
!pip install -q torch-geometric
!pip install -q torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-{TORCH}+cu{CUDA}.html

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, gc, time
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv, global_mean_pool, global_max_pool
from torch_geometric.data import Data, Batch
from torch_geometric.loader import DataLoader
from scipy.spatial.distance import cdist
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from tqdm import tqdm
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

PROJECT_DIR = Path('/content/drive/MyDrive/tactical-gnn')
AIHUB_DIR = PROJECT_DIR / 'data' / 'aihub' / 'training_labels' / '1.동영상분석'
MODEL_DIR = PROJECT_DIR / 'models'
for d in [MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
TACTIC_CLASSES = {
    '빌드업-크로스형': 0, '빌드업-중앙침투형': 1, '빌드업-중앙 침투형': 1,
    '빌드업-측면침투형': 2, '빌드업-측면 침투형': 2,
    '빌드업-롱볼형': 3, '빌드업-롱볼 형': 3,
    '세트피스-코너킥': 4, '세트피스-프리킥': 5,
    '세트피스-스로인': 6, '세트피스-골킥': 7,
    '수비-고위압박': 8, '수비-중위압박': 9, '수비-저위블록': 10,
    '공격전환': 11, '수비전환': 12,
}
NUM_TACTIC_CLASSES = 13
IDX_TO_TACTIC = {v: k for k, v in TACTIC_CLASSES.items()}

POSITION_NAMES = ['GK', 'DF', 'MF', 'FW', 'FP']
POS_TO_IDX = {n: i for i, n in enumerate(POSITION_NAMES)}
POS_TO_ONEHOT = {
    name: [1.0 if i == j else 0.0 for j in range(5)]
    for i, name in enumerate(POSITION_NAMES)
}
POSITION_WEIGHTS = {'GK': 0.6, 'DF': 0.8, 'MF': 1.0, 'FW': 1.2, 'FP': 1.0}

COMPLEMENTARITY_INIT = np.array([
    [0.1, 0.8, 0.4, 0.2, 0.5],
    [0.8, 0.6, 0.9, 0.5, 0.7],
    [0.4, 0.9, 0.5, 0.9, 0.7],
    [0.2, 0.5, 0.9, 0.3, 0.6],
    [0.5, 0.7, 0.7, 0.6, 0.5],
], dtype=np.float32)
COMPLEMENTARITY = COMPLEMENTARITY_INIT

NODE_IN_DIM = 11
EDGE_IN_DIM = 5
HIDDEN_DIM = 64
SAMPLE_RATE = 15    # 전술 구간 내에서 0.5초당 1프레임

print(f'전술 {NUM_TACTIC_CLASSES}클래스, SAMPLE_RATE={SAMPLE_RATE}')

## 2. 경기 메타 로드

In [ ]:
def load_match_meta(match_dir):
    track_path = match_dir / '03' / 'track1.json'
    strategy_path = match_dir / '06' / 'strategy.json'
    if not track_path.exists() or not strategy_path.exists():
        return None
    try:
        with open(strategy_path, encoding='utf-8-sig') as f:
            strategy = json.load(f)
        with open(track_path, encoding='utf-8-sig') as f:
            track = json.load(f)
    except (json.JSONDecodeError, UnicodeDecodeError):
        return None

    fps = track.get('fps', 30)
    n_frames = len(track.get('tag', []))
    pi = track.get('playerInfo', [])
    del track; gc.collect()

    pm = {}
    for p in pi:
        sid = p.get('player_sid')
        if sid is None: continue
        pos = p.get('player_position', 'FP')
        if pos not in POSITION_NAMES: pos = 'FP'
        pm[sid] = {'team': p.get('team', ''), 'position': pos}

    teams = sorted(set(v['team'] for v in pm.values() if v['team'] != '심판'))
    if len(teams) < 2: return None

    tactics = []
    for e in strategy.get('strategyArray', []):
        d1 = e.get('offense_strategy', {}).get('depth1', '')
        tc = TACTIC_CLASSES.get(d1, -1)
        if tc >= 0:
            tactics.append({'start': e['start'], 'end': e['end'], 'tc': tc, 'name': d1})

    if not tactics: return None
    return {
        'match_id': match_dir.name, 'track_path': str(track_path),
        'fps': fps, 'n_frames': n_frames, 'pm': pm,
        'tid': {teams[0]: 0, teams[1]: 1}, 'tactics': tactics,
    }

match_dirs = sorted([d for d in AIHUB_DIR.iterdir() if d.is_dir()])
print(f'경기 폴더: {len(match_dirs)}개')
matches = []
for md in tqdm(match_dirs, desc='Loading meta'):
    m = load_match_meta(md)
    if m: matches.append(m)
print(f'유효 경기: {len(matches)}개')
print(f'총 전술 라벨: {sum(len(m["tactics"]) for m in matches):,}개')

# 전술 분포
td = Counter()
for m in matches:
    for t in m['tactics']: td[t['name']] += 1
print('\n전술 분포:')
for n, c in td.most_common(): print(f'  {n:20s}: {c}')

## 3. 함수 정의

In [ ]:
class TacticalData(Data):
    def __inc__(self, key, value, *args, **kwargs):
        if key == 'link_pairs': return self.x.size(0)
        if key == 'prev_edge_index':
            return self.prev_x.size(0) if hasattr(self, 'prev_x') else 0
        return super().__inc__(key, value, *args, **kwargs)


def extract_frame(tag_entry, pm, tid, img_w=1920.0, img_h=1080.0):
    pos, tids, roles = [], [], []
    for obj in tag_entry.get('data', []):
        sid = obj.get('id'); bbox = obj.get('tag', [])
        if sid is None or len(bbox) < 4: continue
        info = pm.get(sid)
        if info is None or info['team'] == '심판': continue
        t = tid.get(info['team'])
        if t is None: continue
        pos.append([max(0, min(1, (bbox[0]+bbox[2]/2)/img_w)),
                    max(0, min(1, (bbox[1]+bbox[3]/2)/img_h))])
        tids.append(t); roles.append(info['position'])
    if len(pos) < 10: return None
    return {'positions': np.array(pos, dtype=np.float32),
            'team_ids': np.array(tids, dtype=np.int64), 'roles': roles}


def frame_to_graph(positions, team_ids, prev_positions=None, roles=None, k=5):
    n = len(positions); k = min(k, n-1)
    vel = (positions - prev_positions) if (prev_positions is not None and len(prev_positions)==n) else np.zeros_like(positions)
    oh = np.array([POS_TO_ONEHOT.get(r, POS_TO_ONEHOT['FP']) for r in (roles or ['FP']*n)], dtype=np.float32)
    x = np.hstack([positions, vel, team_ids.reshape(-1,1), np.full((n,1),-1.0), oh]).astype(np.float32)
    dm = cdist(positions, positions)
    s, d, ef = [], [], []
    for i in range(n):
        for j in np.argsort(dm[i])[1:k+1]:
            s.append(i); d.append(j)
            ef.append([dm[i,j], np.linalg.norm(vel[i]-vel[j]), float(team_ids[i]==team_ids[j]),
                       positions[j,0]-positions[i,0], positions[j,1]-positions[i,1]])
    return TacticalData(x=torch.tensor(x), edge_index=torch.tensor([s,d], dtype=torch.long),
                        edge_attr=torch.tensor(ef, dtype=torch.float32))


def synergy_targets(positions, team_ids, roles, n_samples=20, seed=0):
    n = len(positions); roles = roles or ['FP']*n
    dm = cdist(positions, positions); mx = dm.max()+1e-8
    rng = np.random.default_rng(seed)
    same = [(a,b) for a in range(n) for b in range(a+1,n) if team_ids[a]==team_ids[b]]
    diff = [(a,b) for a in range(n) for b in range(a+1,n) if team_ids[a]!=team_ids[b]]
    k = min(len(same), len(diff), n_samples)
    if k == 0: return torch.zeros((0,2), dtype=torch.long), torch.zeros(0)
    pairs, labels = [], []
    for a,b in [same[i] for i in rng.choice(len(same), k, replace=False)]:
        c = COMPLEMENTARITY[POS_TO_IDX.get(roles[a],4), POS_TO_IDX.get(roles[b],4)]
        pairs.append([a,b]); labels.append(float(c * max(0, 1-dm[a,b]/mx)))
    for a,b in [diff[i] for i in rng.choice(len(diff), k, replace=False)]:
        pairs.append([a,b]); labels.append(0.0)
    return torch.tensor(pairs, dtype=torch.long), torch.tensor(labels, dtype=torch.float32)

print('함수 준비 완료')

## 4. 데이터셋 구축 — 전술 구간 내 프레임만 추출

핵심 변경: 전체 프레임 샘플링이 아니라, `strategy.json`의 `start~end` 범위 내 프레임만 추출
→ 라벨률 ~100%, 불필요한 unlabeled 프레임 제거 → 메모리 절약 + 학습 효율

In [ ]:
all_graphs = []
label_counter = Counter()
seed_counter = 0

for match in tqdm(matches, desc='Building graphs'):
    try:
        with open(match['track_path'], encoding='utf-8-sig') as f:
            tags = json.load(f).get('tag', [])
    except Exception:
        continue

    # frame_num → tag index 매핑 (빠른 검색)
    frame_to_idx = {}
    for i, t in enumerate(tags):
        fn = int(t.get('frame', i))
        frame_to_idx[fn] = i

    pm = match['pm']
    tid = match['tid']
    prev_graph = None
    prev_pos = None

    # 전술 구간별로 프레임 추출
    for tactic in match['tactics']:
        tc = tactic['tc']
        start_f = tactic['start']
        end_f = tactic['end']

        # 구간 내 프레임 샘플링
        sampled_frames = list(range(start_f, end_f + 1, SAMPLE_RATE))
        if not sampled_frames:
            sampled_frames = [start_f]  # 최소 1프레임

        for fn in sampled_frames:
            idx = frame_to_idx.get(fn)
            if idx is None:
                # 정확한 프레임이 없으면 가장 가까운 프레임 찾기
                closest = min(frame_to_idx.keys(), key=lambda x: abs(x - fn), default=None)
                if closest is not None and abs(closest - fn) < SAMPLE_RATE:
                    idx = frame_to_idx[closest]
                else:
                    continue

            frame = extract_frame(tags[idx], pm, tid)
            if frame is None:
                prev_pos = None; prev_graph = None
                continue

            positions = frame['positions']
            team_ids = frame['team_ids']
            roles = frame['roles']

            g = frame_to_graph(positions, team_ids, prev_pos, roles)
            g.y = torch.tensor([tc], dtype=torch.long)
            g.y_mask = torch.tensor([1.0])  # 모두 라벨 있음
            g.change_label = torch.tensor([0.0])

            seed_counter += 1
            lp, ll = synergy_targets(positions, team_ids, roles, seed=seed_counter)
            g.link_pairs = lp; g.link_labels = ll

            if prev_graph is not None:
                g.prev_x = prev_graph.x.clone()
                g.prev_edge_index = prev_graph.edge_index.clone()
                g.prev_edge_attr = prev_graph.edge_attr.clone()
                g.prev_num_nodes = torch.tensor([prev_graph.num_nodes])
                g.has_prev = torch.tensor([1.0])
            else:
                g.prev_x = g.x.clone()
                g.prev_edge_index = g.edge_index.clone()
                g.prev_edge_attr = g.edge_attr.clone()
                g.prev_num_nodes = torch.tensor([g.num_nodes])
                g.has_prev = torch.tensor([0.0])

            prev_graph = g; prev_pos = positions
            all_graphs.append(g)
            label_counter[tc] += 1

    del tags; gc.collect()

# 변화점 라벨
ch = 0
for i in range(1, len(all_graphs)):
    if all_graphs[i].y.item() != all_graphs[i-1].y.item():
        all_graphs[i].change_label = torch.tensor([1.0])
        ch += 1

print(f'\n=== 데이터셋 ===')
print(f'  그래프: {len(all_graphs)}개 (100% 라벨)')
print(f'  변화점: {ch}개 ({ch/max(len(all_graphs),1)*100:.1f}%)')
print(f'\n  클래스별 분포:')
for tc, cnt in sorted(label_counter.items()):
    print(f'    {IDX_TO_TACTIC.get(tc, f"c{tc}"):20s}: {cnt}')

In [ ]:
# Train / Val / Test
train_g, test_g = train_test_split(all_graphs, test_size=0.15, random_state=42, stratify=[g.y.item() for g in all_graphs])
train_g, val_g = train_test_split(train_g, test_size=0.12, random_state=42, stratify=[g.y.item() for g in train_g])
print(f'Train: {len(train_g)}, Val: {len(val_g)}, Test: {len(test_g)}')

# 클래스 가중치 계산 (편향 보정)
train_labels = [g.y.item() for g in train_g]
train_dist = Counter(train_labels)
total = len(train_labels)
n_classes_present = len(train_dist)
class_weights = torch.ones(NUM_TACTIC_CLASSES, device=device)
for tc, cnt in train_dist.items():
    class_weights[tc] = total / (n_classes_present * cnt)
print(f'\n클래스 가중치:')
for tc, w in enumerate(class_weights):
    if w != 1.0:
        print(f'  {IDX_TO_TACTIC.get(tc, f"c{tc}"):20s}: {w:.2f}')

BS = 32
train_loader = DataLoader(train_g, batch_size=BS, shuffle=True)
val_loader = DataLoader(val_g, batch_size=BS)
test_loader = DataLoader(test_g, batch_size=BS)

## 5. 모델 (UW 클램핑 추가)

In [ ]:
class SharedGATBackbone(nn.Module):
    def __init__(self, in_dim, hidden_dim, heads=4, edge_dim=5):
        super().__init__()
        self.input_norm = nn.BatchNorm1d(in_dim)
        self.conv1 = GATv2Conv(in_dim, hidden_dim, heads=heads, edge_dim=edge_dim, concat=True)
        self.norm1 = nn.LayerNorm(hidden_dim * heads)
        self.conv2 = GATv2Conv(hidden_dim * heads, hidden_dim, heads=1, edge_dim=edge_dim, concat=False)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.res_proj = nn.Linear(in_dim, hidden_dim)
    def forward(self, x, edge_index, edge_attr, batch):
        x0 = self.input_norm(x)
        h = F.elu(self.norm1(self.conv1(x0, edge_index, edge_attr)))
        h = F.dropout(h, p=0.1, training=self.training)
        h = F.elu(self.norm2(self.conv2(h, edge_index, edge_attr)))
        ne = h + self.res_proj(x0)
        return ne, torch.cat([global_mean_pool(ne, batch), global_max_pool(ne, batch)], dim=-1)

class TaskAdapter(nn.Module):
    def __init__(self, i, o):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(i, o), nn.GELU(), nn.LayerNorm(o))
    def forward(self, x): return self.net(x)

class LearnableComplementarity(nn.Module):
    def __init__(self, init_matrix):
        super().__init__()
        t = torch.tensor(init_matrix, dtype=torch.float32).clamp(0.01, 0.99)
        self.C_logits = nn.Parameter(torch.log(t / (1.0 - t)))
    def forward(self, pi, pj, dist):
        return torch.sigmoid(self.C_logits)[pi, pj] * torch.exp(-dist)
    def get_matrix(self):
        with torch.no_grad(): return torch.sigmoid(self.C_logits).cpu().numpy()

class LinkPredictionHead(nn.Module):
    def __init__(self, d, init_m):
        super().__init__()
        self.comp = LearnableComplementarity(init_m)
        self.adapter = TaskAdapter(d, d)
        self.mlp = nn.Sequential(nn.Linear(d*2,d), nn.GELU(), nn.Dropout(0.2),
                                 nn.Linear(d,d//2), nn.GELU(), nn.Linear(d//2,1))
    def forward(self, ne, pairs, pi=None, dist=None):
        if pairs.numel() == 0: return torch.zeros(0, device=ne.device)
        a = self.adapter(ne)
        es = torch.sigmoid(self.mlp(torch.cat([a[pairs[:,0]], a[pairs[:,1]]], dim=-1))).squeeze(-1)
        if pi is not None and dist is not None:
            cs = self.comp(pi[pairs[:,0]], pi[pairs[:,1]], dist)
            return 0.5 * es + 0.5 * cs
        return es

class TacticHead(nn.Module):
    def __init__(self, gd, nc):
        super().__init__()
        self.adapter = TaskAdapter(gd, gd//2)
        self.mlp = nn.Sequential(nn.Linear(gd//2, gd//2), nn.GELU(), nn.Dropout(0.2), nn.Linear(gd//2, nc))
    def forward(self, g): return self.mlp(self.adapter(g))

class ChangeHead(nn.Module):
    def __init__(self, gd):
        super().__init__()
        self.adapter = TaskAdapter(gd*3, gd)
        self.mlp = nn.Sequential(nn.Linear(gd, gd//2), nn.GELU(), nn.Dropout(0.2), nn.Linear(gd//2, 1))
    def forward(self, t, p):
        return self.mlp(self.adapter(torch.cat([t, p, t-p], dim=-1)))

class UncertaintyWeighting(nn.Module):
    """log_vars 클램핑으로 발산 방지."""
    def __init__(self, n):
        super().__init__()
        self.log_vars = nn.Parameter(torch.zeros(n))
    def forward(self, *losses):
        total = torch.tensor(0.0, device=self.log_vars.device)
        clamped = self.log_vars.clamp(-2.0, 2.0)  # FIX: 발산 방지
        for i, l in enumerate(losses):
            total = total + torch.exp(-clamped[i]) * l + clamped[i]
        return total

class TacticalGNN(nn.Module):
    def __init__(self, nid=11, hd=64, nc=13, gh=4, ed=5, ci=None):
        super().__init__()
        gd = hd * 2
        self.backbone = SharedGATBackbone(nid, hd, gh, ed)
        self.head_link = LinkPredictionHead(hd, ci if ci is not None else COMPLEMENTARITY_INIT)
        self.head_tactic = TacticHead(gd, nc)
        self.head_change = ChangeHead(gd)
        self.uw = UncertaintyWeighting(3)
    def _pb(self, pnn):
        p = pnn.view(-1)
        return torch.repeat_interleave(torch.arange(p.size(0), device=p.device), p)
    def forward(self, data):
        ne, ge = self.backbone(data.x, data.edge_index, data.edge_attr, data.batch)
        _, pe = self.backbone(data.prev_x, data.prev_edge_index, data.prev_edge_attr, self._pb(data.prev_num_nodes))
        tl = self.head_tactic(ge)
        cl = self.head_change(ge, pe)
        ls = torch.zeros(0, device=ne.device)
        if hasattr(data, 'link_pairs') and data.link_pairs.numel() > 0:
            pi = data.x[:, 6:11].argmax(dim=1)
            pr = data.link_pairs
            ds = torch.norm(data.x[pr[:,0],:2] - data.x[pr[:,1],:2], dim=1)
            ls = self.head_link(ne, pr, pi, ds)
        return {'ne': ne, 'ge': ge, 'tl': tl, 'cl': cl, 'ls': ls}

model = TacticalGNN(nid=NODE_IN_DIM, hd=HIDDEN_DIM, nc=NUM_TACTIC_CLASSES,
                    ed=EDGE_IN_DIM, ci=COMPLEMENTARITY_INIT).to(device)
print(f'TacticalGNN: {sum(p.numel() for p in model.parameters()):,} params')

## 6. 학습

In [ ]:
cr = sum(g.change_label.item() for g in all_graphs) / max(len(all_graphs), 1)
cpw = min(max((1.0 - cr) / max(cr, 0.01), 1.0), 20.0)  # 최대 20으로 제한

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=80, eta_min=1e-5)
crit_t = nn.CrossEntropyLoss(weight=class_weights, reduction='none')
crit_l = nn.MSELoss()
crit_c = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([cpw], device=device), reduction='none')

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    s = {'loss':0, 'tc':0, 'tn':0, 'lm':0, 'ln':0}
    for batch in loader:
        batch = batch.to(device)
        if train: optimizer.zero_grad()
        with torch.set_grad_enabled(train):
            o = model(batch)
            y = batch.y.view(-1); mb = batch.y_mask.view(-1)
            cl = batch.change_label.view(-1); mc = batch.has_prev.view(-1)
            lt = (crit_t(o['tl'], y) * mb).sum() / mb.sum().clamp(min=1)
            lc = (crit_c(o['cl'].view(-1), cl) * mc).sum() / mc.sum().clamp(min=1)
            ll = torch.tensor(0.0, device=device)
            if o['ls'].numel() > 0:
                ll = crit_l(o['ls'], batch.link_labels.to(device))
                s['lm'] += ll.item() * len(o['ls']); s['ln'] += len(o['ls'])
            loss = model.uw(lt, ll, lc)
        if train:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        s['loss'] += loss.item() * batch.num_graphs
        v = mb > 0
        if v.any():
            s['tc'] += (o['tl'][v].argmax(1)==y[v]).sum().item()
            s['tn'] += v.sum().item()
    return s['loss']/max(len(loader.dataset),1), s['tc']/max(s['tn'],1), s['lm']/max(s['ln'],1)

print(f'변화점: {cr:.3f}, pos_weight: {cpw:.1f}')

In [ ]:
EPOCHS = 80
best_val_acc = 0.0
history = {k: [] for k in ['tl','vl','tt','vt','tlk','vlk']}

print('=== Training (Real Data v2) ===\n')
for epoch in range(1, EPOCHS + 1):
    tl, tt, tlk = run_epoch(train_loader, True)
    vl, vt, vlk = run_epoch(val_loader, False)
    scheduler.step()
    history['tl'].append(tl); history['vl'].append(vl)
    history['tt'].append(tt); history['vt'].append(vt)
    history['tlk'].append(tlk); history['vlk'].append(vlk)
    if vt > best_val_acc:
        best_val_acc = vt
        torch.save(model.state_dict(), str(MODEL_DIR / 'best_real_v2.pt'))
    if epoch % 5 == 0 or epoch == 1:
        uw = torch.exp(-model.uw.log_vars.clamp(-2,2)).detach().cpu().numpy()
        print(f'Epoch {epoch:3d} | Loss {tl:.4f}/{vl:.4f} | Tac {tt:.3f}/{vt:.3f} | '
              f'Link {tlk:.4f}/{vlk:.4f} | UW [{uw[0]:.2f},{uw[1]:.2f},{uw[2]:.2f}] | Best {best_val_acc:.3f}')

print(f'\nComplete. Best Val Acc: {best_val_acc:.3f}')

## 7. 평가

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(history['tl'], label='Train'); axes[0].plot(history['vl'], label='Val')
axes[0].set_title('Loss'); axes[0].legend()
axes[1].plot(history['tt'], label='Train'); axes[1].plot(history['vt'], label='Val')
axes[1].set_title('Tactic Acc'); axes[1].legend()
axes[2].plot(history['tlk'], label='Train'); axes[2].plot(history['vlk'], label='Val')
axes[2].set_title('Link MSE'); axes[2].legend()
plt.tight_layout(); plt.savefig(str(PROJECT_DIR / 'curves_real_v2.png'), dpi=150); plt.show()

In [ ]:
model.load_state_dict(torch.load(str(MODEL_DIR / 'best_real_v2.pt')))
tl, ta, tlk = run_epoch(test_loader, False)
print(f'=== Test (Real Data v2) ===')
print(f'Loss: {tl:.4f}, Tactic Acc: {ta:.3f}, Link MSE: {tlk:.4f}')

preds, labels = [], []
model.eval()
with torch.no_grad():
    for b in test_loader:
        b = b.to(device)
        o = model(b)
        m = b.y_mask.view(-1) > 0
        if m.any():
            preds.extend(o['tl'][m].argmax(1).cpu().numpy())
            labels.extend(b.y.view(-1)[m].cpu().numpy())

pc = sorted(set(labels) | set(preds))
print('\n=== Classification Report ===')
print(classification_report(labels, preds, labels=pc,
      target_names=[IDX_TO_TACTIC.get(i, f'c{i}') for i in pc], zero_division=0))

## 8. 상보성 행렬 + 저장

In [ ]:
learned = model.head_link.comp.get_matrix()
uw = torch.exp(-model.uw.log_vars.clamp(-2,2)).detach().cpu().numpy()

ckpt = {
    'model_state_dict': model.state_dict(),
    'config': {'node_in_dim': NODE_IN_DIM, 'edge_dim': EDGE_IN_DIM,
               'hidden_dim': HIDDEN_DIM, 'num_classes': NUM_TACTIC_CLASSES},
    'complementarity_init': COMPLEMENTARITY_INIT.tolist(),
    'complementarity_learned': learned.tolist(),
    'task_weights': {'tactic': float(uw[0]), 'link': float(uw[1]), 'change': float(uw[2])},
    'history': history, 'best_val_acc': best_val_acc,
    'data': f'AI Hub {len(matches)} matches, {len(all_graphs)} labeled graphs',
    'label_dist': dict(label_counter),
}
torch.save(ckpt, str(MODEL_DIR / 'tactical_gnn_real_v2.pt'))

print('=== 상보성 행렬 ===')
print(f'포지션: {POSITION_NAMES}\n')
print('초기값:'); print(COMPLEMENTARITY_INIT.round(2))
print(f'\n학습 후:'); print(learned.round(3))
print(f'\n차이:'); print(np.array2string(learned - COMPLEMENTARITY_INIT, precision=3, sign='+'))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, mat, title in [(axes[0], COMPLEMENTARITY_INIT, 'Initial'),
                        (axes[1], learned, 'Learned'),
                        (axes[2], learned - COMPLEMENTARITY_INIT, 'Difference')]:
    vmin = -0.3 if 'Diff' in title else 0
    vmax = 0.3 if 'Diff' in title else 1
    im = ax.imshow(mat, cmap='RdYlGn', vmin=vmin, vmax=vmax)
    ax.set_xticks(range(5)); ax.set_xticklabels(POSITION_NAMES)
    ax.set_yticks(range(5)); ax.set_yticklabels(POSITION_NAMES)
    ax.set_title(title)
    for i in range(5):
        for j in range(5):
            ax.text(j, i, f'{mat[i,j]:.2f}', ha='center', va='center', fontsize=10)
    plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(str(PROJECT_DIR / 'complementarity_real_v2.png'), dpi=150)
plt.show()

print(f'\n저장: {MODEL_DIR / "tactical_gnn_real_v2.pt"}')
print(f'Task weights: tactic={uw[0]:.3f}, link={uw[1]:.3f}, change={uw[2]:.3f}')

# v1 vs v2 비교
print(f'\n=== v1 vs v2 ===')
print(f'  v1: 13,254 graphs (4.8% labeled), Val Acc 75.0%, Test Acc 61.7%')
print(f'  v2: {len(all_graphs)} graphs (100% labeled), Val Acc {best_val_acc:.3f}, Test Acc {ta:.3f}')